# 01 — EDA HAM10000

Khám phá dữ liệu cho báo cáo (PLAN_BTL.md mục 5.3). Chỉ đọc metadata, manifest và ảnh; **không** dùng kết quả test. Chạy từ thư mục gốc repo hoặc từ `notebooks/`. Hình được lưu vào `reports/figures/eda_*.png`.

In [ ]:
from pathlib import Path
import hashlib, json
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA = ROOT / "data/raw/ham10000"
FIG = ROOT / "reports/figures"; FIG.mkdir(parents=True, exist_ok=True)
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
meta = pd.read_csv(DATA / "HAM10000_metadata.csv")
manifest = json.loads((ROOT / "data/splits/manifest.json").read_text(encoding="utf-8"))
meta["split"] = meta["image_id"].map(manifest["splits"])
print(meta.shape); meta.head()

## 1. Phân phối lớp (mất cân bằng)

In [ ]:
counts = meta["dx"].value_counts().reindex(CLASSES)
table = pd.DataFrame({"images": counts, "percent": (100 * counts / counts.sum()).round(2),
                      "lesions": meta.groupby("dx")["lesion_id"].nunique().reindex(CLASSES)})
print("imbalance ratio nv/df =", round(counts.max() / counts.min(), 1))
ax = counts.plot.bar(color="#4C72B0", figsize=(6, 3.2)); ax.set_ylabel("images"); ax.set_yscale("log")
ax.set_title("HAM10000 class distribution (log scale)"); plt.tight_layout(); plt.savefig(FIG / "eda_class_distribution.png", dpi=200)
table

## 2. Nhiều ảnh cho cùng một tổn thương → lý do chia theo `lesion_id`

In [ ]:
per_lesion = meta.groupby("lesion_id").size()
print("images:", len(meta), "| lesions:", per_lesion.size,
      "| images belonging to multi-image lesions:", int(per_lesion[per_lesion > 1].sum()))
print(per_lesion.value_counts().sort_index().rename("lesions with k images"))
print("lesions with >1 label:", int((meta.groupby("lesion_id")["dx"].nunique() > 1).sum()))

## 3. Metadata: tuổi, giới, vị trí

In [ ]:
print("age missing:", int(meta["age"].isna().sum()), "| age == 0:", int((meta["age"] == 0).sum()))
print(meta["sex"].value_counts(dropna=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
meta.boxplot(column="age", by="dx", ax=axes[0]); axes[0].set_title("age by class"); axes[0].set_xlabel("")
pd.crosstab(meta["localization"], meta["dx"], normalize="columns").plot.bar(stacked=False, ax=axes[1], width=0.8)
axes[1].set_title("localization share within each class"); axes[1].legend(fontsize=7, ncol=4)
plt.suptitle(""); plt.tight_layout(); plt.savefig(FIG / "eda_metadata.png", dpi=200)
pd.crosstab(meta["dx"], meta["dx_type"])

Metadata mang thông tin về lớp (tuổi, vị trí khác nhau giữa các lớp) → động cơ cho việc dùng metadata. `dx_type` (cách xác nhận chẩn đoán) **không** dùng làm đầu vào vì đó là thông tin rò rỉ về nhãn.

## 4. Phép chia train/validation/test (theo manifest đã commit)

In [ ]:
split_table = pd.crosstab(meta["split"], meta["dx"]).reindex(["train", "validation", "test"])[CLASSES]
split_table["total"] = split_table.sum(axis=1)
split_table["lesions"] = meta.groupby("split")["lesion_id"].nunique()
leak = meta.groupby("lesion_id")["split"].nunique().gt(1).sum()
print("lesions in >1 split:", int(leak))
split_table

Lưu ý cho báo cáo: test chỉ có rất ít ảnh `df`/`vasc` → F1 của các lớp này có phương sai lớn; báo cáo mean ± std qua seed (và có thể bootstrap CI).

## 5. Ảnh ví dụ mỗi lớp

In [ ]:
index = {p.stem: p for d in sorted(DATA.glob("HAM10000_images_part_*")) for p in d.glob("*.jpg")}
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(8, 14))
for row, dx in enumerate(CLASSES):
    sample = meta[(meta["dx"] == dx) & (meta["split"] == "train")].sample(4, random_state=0)
    for col, image_id in enumerate(sample["image_id"]):
        axes[row, col].imshow(Image.open(index[image_id])); axes[row, col].axis("off")
    axes[row, 0].set_title(dx, loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(FIG / "eda_examples.png", dpi=150)

## 6. Kích thước ảnh và ảnh trùng hoàn toàn (hash)

In [ ]:
sizes = Counter(Image.open(p).size for p in list(index.values())[:500])
print("image sizes (first 500):", sizes)
hashes = {}
for image_id, path in index.items():
    hashes.setdefault(hashlib.md5(path.read_bytes()).hexdigest(), []).append(image_id)
dups = [ids for ids in hashes.values() if len(ids) > 1]
print("exact duplicate files:", len(dups), dups[:5])